In [1]:
# Dependencies
import pickle
import numpy as np

import sys
sys.path.append("/home/rguo_hpc/myfolder/mocap")
from datasets.transform import NormalizeConfig, ViewInvariant

In [3]:
# Load data
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    data_fmr1 = pickle.load(file)

feats = []
for mouse in data_fmr1.keys():
    num_seq = len(data_fmr1[mouse]["ratgen"])
    ratgen  = int(data_fmr1[mouse]["ratgen"][0])
    feat = np.array(data_fmr1[mouse]["m1"]).transpose(0, 1, 3, 2)     # (3, 90000, 3, 23)
    feat = np.delete(feat, [5, 10, 14, 18, 22], axis=-2)
    feats.append(feat)

print(len(feats), feats[0].shape)

8 (3, 90000, 18, 3)


In [4]:
# Reshape to (N, 50, 18, 3)
feats = np.concatenate(feats, axis = 0).reshape(-1, 50, 18, 3)
print(feats.shape)

(43200, 50, 18, 3)


In [ ]:
# view invariant -> augment -> normalize
vi = ViewInvariant(index_frame = 25, left_idx = 12, right_idx = 15)
feats_norm = np.zeros(feats.shape)

for i in range(len(feats)): 
    # View invariant
    feats_norm[i], _, _ =  vi(feats[i], x_supp=(),)

In [ ]:
np.save("clips.npy", feats_norm)

# Analysis

In [16]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, silhouette_score

L = 4500
tp = 1
l = int(L/tp) # after patch
N = 20
sample_frequency = 9
D = 192
fmr1_fold_1 = {"train":[402, 404, 405, 406, 407, 408], "valid": [401, 403]}

In [6]:
# load original data 
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    result = pickle.load(file)

for mouse in result.keys():     # result.keys(): [408, 407, 404, 403, 402, 406, 405, 401]
    num_seq = len(result[mouse]["ratgen"])
    ratgen  = int(result[mouse]["ratgen"][0])
    result[mouse]["ratgen"] = [ratgen for i in range(num_seq * N,)]
    ratid = result[mouse]["ratid"][0]
    result[mouse]["ratid"] = [ratid for i in range(num_seq * N,)]
    
    # llac    
    llac = np.array(result[mouse]["llac"])
    llac = np.squeeze(llac, axis=2)
    llac = llac.reshape(-1, L,)
    if tp > 1:
        grouped_llac = llac.reshape(-1, l, tp) # choose the mode label of each patch 
        result[mouse]["llac"] = mode(grouped_llac, axis=2, keepdims=False).mode.astype(np.uint8)
    else:
        result[mouse]["llac"] = llac
    
    # hlac
    hlac = np.array(result[mouse]["hlac"]) # (3, 90000, 1)
    hlac = np.squeeze(hlac, axis=2)
    hlac = hlac.reshape(-1, L,)
    if tp > 1:
        grouped_hlac = hlac.reshape(-1, l, 3)
        result[mouse]["hlac"] = mode(grouped_hlac, axis=2, keepdims=False).mode.astype(np.uint8)
    else:
        result[mouse]["hlac"] = hlac
    
    del result[mouse]["m1"]

In [8]:
# Train: mouse, genotype, hlac, llac
mouse_tr, gen_tr, hlac_tr, llac_tr = [], [], [], []
for mouse_id in fmr1_fold_1["train"]:
    mouse_tr.append(result[mouse_id]["ratid"])
    gen_tr.append(result[mouse_id]["ratgen"])
    hlac_tr.append(result[mouse_id]["hlac"])
    llac_tr.append(result[mouse_id]["llac"])
    
mouse_tr = np.repeat(np.concatenate(mouse_tr), l)[::sample_frequency]
gen_tr =  np.repeat(np.concatenate(gen_tr), l)[::sample_frequency]
hlac_tr = np.concatenate(hlac_tr)[:,::sample_frequency]
llac_tr = np.concatenate(llac_tr)[:,::sample_frequency]

# Val:  mouse, genotype, hlac, llac
mouse_val, gen_val, hlac_val, llac_val = [], [], [], []
for mouse_id in fmr1_fold_1["valid"]:
    mouse_val.append(result[mouse_id]["ratid"])
    gen_val.append(result[mouse_id]["ratgen"])
    hlac_val.append(result[mouse_id]["hlac"])
    llac_val.append(result[mouse_id]["llac"]) 
    
mouse_val = np.repeat(np.concatenate(mouse_val), l)[::sample_frequency]
gen_val = np.repeat(np.concatenate(gen_val), l)[::sample_frequency]
hlac_val = np.concatenate(hlac_val)[:,::sample_frequency]
llac_val = np.concatenate(llac_val)[:,::sample_frequency]

hlac_tr = hlac_tr.reshape(hlac_tr.size, )
hlac_val = hlac_val.reshape(hlac_val.size, )

llac_tr = llac_tr.reshape(llac_tr.size, )
llac_val = llac_val.reshape(llac_val.size, )

In [34]:
tr_feats = np.load("../swav_output/new_representations_train.npy")[:, 25:4525][:,::sample_frequency]
val_feats = np.load("../swav_output/new_representations_valid.npy")[:, 25:4525][:,::sample_frequency]
tr_feats = tr_feats.reshape(-1, 128)
val_feats = val_feats.reshape(-1, 128)

In [35]:
model = LogisticRegression(max_iter=500, multi_class='multinomial')
#model = RandomForestClassifier()
model.fit(tr_feats, hlac_tr)
# Predict
y_pred = model.predict(val_feats)

# Evaluate
print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))

/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.6768

Classification Report:
               precision    recall  f1-score   support

           1       0.66      0.70      0.68     11792
           2       0.63      0.59      0.61     16949
           3       0.69      0.86      0.77      2183
           4       0.66      0.28      0.39      2158
           5       0.41      0.33      0.37       961
           6       0.79      0.96      0.87      6165
           7       0.66      0.66      0.66     12772
           8       0.76      0.74      0.75      7020

    accuracy                           0.68     60000
   macro avg       0.66      0.64      0.64     60000
weighted avg       0.67      0.68      0.67     60000



original pretrain 15 epoch: 0.7310 
pretrain 10 epochs + finetune last 2 blocks, 128 prototypes
5 epoch: 0.7338
10 epoch: 0.74307
15 epoch: 0.74315
20 epochs 0.7422